# 추가 실험 1 — 파일럿 A (245문항 기준)

P2(TD+CD+SR) · gpt-6-luna · one-call · **에러 픽싱 없음**

| 항목 | 값 |
|---|---|
| 문항 | `questions_245.csv` 에서 **무작위 10개** (seed 고정, 목록 저장) |
| 조건 | `reasoning_effort` = low / medium / high × **T=1.0** |
| 반복 | 문항·조건당 **3회** |
| 호출 | 10 × 3 × 3 = **90** |
| 지표 | 정답률 · 실행 오류 건수 · 출력/사고 토큰 · 응답시간 |


In [ ]:
# 0) 경로
import sys, pathlib
LAB = pathlib.Path.cwd()
sys.path.insert(0, str(LAB))
sys.path.insert(0, str(LAB.parent))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("LAB  =", LAB)
print("repo =", LAB.parent, "| core/:", (LAB.parent/"core").exists(), "| data/:", (LAB.parent/"data").exists())

In [2]:
# 1) 문항 · 기준 정답 (245)
import exp1_lib as L

qs = pd.read_csv(LAB/"questions_245.csv", encoding="utf-8-sig")
qs["n"] = qs.index + 1

ref = pd.read_csv(LAB/"answers_ref_245.csv", encoding="utf-8-sig")   # n, answer_ref, ...
assert len(ref) == len(qs) and ref["answer_ref"].notna().all(), "기준정답 누락"
print(f"문항 {len(qs)} · 기준정답 {ref['answer_ref'].notna().sum()} · gold 불일치 {(ref['match']=='X').sum()}")

K_Q, SEED = 10, L.SEED_DEFAULT
pilot = L.pick_questions(qs, k=K_Q, seed=SEED)
print(f"추출 {len(pilot)}문항 (seed={SEED}) → {L.RESULT_DIR}/pilot_questions_seed{SEED}.csv")
pilot[["n", "dataset", "question"]].head(10)


문항 100 · 기준정답 100 · gold 불일치 3


,n,dataset,A,B,question
0,1,001_Forbes,A0,B0,What is the age of the youngest billionaire?
4,5,003_Love,A0,B0,What is the average gross annual salary?
11,12,019_Aircraft,A0,B1,What's the highest number of occupants recorde...
17,18,035_Billboard,A4,B0,What's the rank of the song with the longest l...
25,26,045_Predict,A2,B0,How many songs were released in the year with ...
40,41,066_IBM_HR,A0,B0,What is the maximum years someone has been at ...
51,52,069_Taxonomy,A1,B2,How many entries belong to the third or fourth...
59,60,072_Admissions,A4,B0,What is the rating given by the college they a...
76,77,074_Lift,A2,B2,What is the least amount of kilos lifted in th...
86,87,078_Fires,A4,B2,"If the year starts in January, what is the num..."


In [3]:
# 2) 클라이언트 + 파라미터
MODEL = "gpt-6-luna"
N_REP = 3          # 문항·조건당 반복 (기존 2회 → 3회)
client = L.make_client()

# L.probe_capabilities(client, MODEL)   # <- 필요할 때만 주석 해제


In [4]:
# 3) 프롬프트 1건 미리보기
r = pilot.iloc[0]
prompt = L.build_prompt(r.to_dict())
try:
    import tiktoken
    n_tok, note = len(tiktoken.get_encoding("o200k_base").encode(prompt)), "실측"
except ImportError:
    n_tok, note = int(len(prompt)/3.89), "근사"
print(f"n={int(r['n'])} | {r['dataset']} | {len(prompt):,}자 ≈ {n_tok:,}토큰 [{note}]")
print(prompt[-300:])

n=1 | 001_Forbes | 77,012자 ≈ 19,796토큰 [실측]
 in the same Omaha, Nebrask...
    """


    df.columns = ['rank', 'personName', 'age', 'finalWorth', 'category', 'source', 'country', 'state', 'city', 'organization', 'selfMade', 'gender', 'birthDate', 'title', 'philanthropyScore', 'bio', 'about']
    
    # The columns used to answer the question:


## 갈래 A — 추론 켬

In [5]:
# 3A) 파일럿 A 실행 (low/medium/high × T=1.0 × 3회 = 90회)
#     max_completion_tokens=2000 — 사고 토큰이 본문을 밀어내지 않게
resA = L.run_branch(client, pilot, ref,
                    out_csv="pilot_A_245.csv",
                    branch="A", n_rep=N_REP, model=MODEL)


[A_elow_T1] max_tokens=2000
  A_elow_T1 n=  1 rep=1 O exec=ok fr=stop     out=90 rea=46
  A_elow_T1 n=  1 rep=2 O exec=ok fr=stop     out=135 rea=37
  A_elow_T1 n=  5 rep=1 X exec=ok fr=stop     out=239 rea=194
  A_elow_T1 n=  5 rep=2 X exec=ok fr=stop     out=511 rea=36
  A_elow_T1 n= 12 rep=1 O exec=ok fr=stop     out=297 rea=100
  A_elow_T1 n= 12 rep=2 X exec=ok fr=stop     out=298 rea=80
  A_elow_T1 n= 18 rep=1 X exec=ok fr=stop     out=119 rea=46
  A_elow_T1 n= 18 rep=2 O exec=ok fr=stop     out=116 rea=38
  A_elow_T1 n= 26 rep=1 X exec=ok fr=stop     out=172 rea=53
  A_elow_T1 n= 26 rep=2 X exec=ok fr=stop     out=156 rea=37
  A_elow_T1 n= 41 rep=1 O exec=ok fr=stop     out=110 rea=62
  A_elow_T1 n= 41 rep=2 O exec=ok fr=stop     out=111 rea=58
  A_elow_T1 n= 52 rep=1 O exec=ok fr=stop     out=190 rea=115
  A_elow_T1 n= 52 rep=2 O exec=ok fr=stop     out=189 rea=136
  A_elow_T1 n= 60 rep=1 X exec=ok fr=stop     out=148 rea=50
  A_elow_T1 n= 60 rep=2 X exec=ok fr=stop     out=147 

In [6]:
# 4A) 요약 — 정답률 / 실행 오류 건수 / 토큰 / 응답시간
L.summarize(resA, by_condition=True)


,condition,호출 수,호출 실패,finish=length,본문 비어있음,실행 성공률 %,서명 추출 %,정답률 %,값 일관성 %,의도 일관성 %,출력토큰 평균,사고토큰 평균,캐시hit 평균,fingerprint 수,비용 USD
0,A_elow_T1,20,0,0,0,100.0,100.0,60.0,85.7,90.0,192.9,84.0,9917.0,0,0.0238
1,A_emedium_T1,20,0,0,0,95.0,95.0,35.0,58.3,90.0,216.2,120.6,9917.0,0,0.0240
2,A_ehigh_T1,20,0,0,0,95.0,95.0,20.0,66.7,90.0,365.0,257.9,9917.0,0,0.0255


## 갈래 B — 이번에는 실행하지 않음


In [13]:
# 4B) 파일럿 B 실행 (effort none × T=0/0.2/1.0 × 2회 = 60회)
resB = L.run_branch(client, pilot, ref,
                    out_csv=LAB/"pilot_B_runs.csv",
                    branch="B", n_rep=N_REP, model=MODEL)

[B_enone_T0] max_tokens=300
  B_enone_T0 n=  1 rep=1 X exec=ok fr=stop     out=104 rea=0
  B_enone_T0 n=  1 rep=2 O exec=ok fr=stop     out=80 rea=0
  B_enone_T0 n=  5 rep=1 O exec=ok fr=stop     out=31 rea=0
  B_enone_T0 n=  5 rep=2 O exec=ok fr=stop     out=31 rea=0
  B_enone_T0 n= 12 rep=1 X exec=ok fr=stop     out=233 rea=0
  B_enone_T0 n= 12 rep=2 X exec=ok fr=stop     out=245 rea=0
  B_enone_T0 n= 18 rep=1 O exec=ok fr=stop     out=67 rea=0
  B_enone_T0 n= 18 rep=2 O exec=ok fr=stop     out=68 rea=0
  B_enone_T0 n= 26 rep=1 X exec=ok fr=stop     out=40 rea=0
  B_enone_T0 n= 26 rep=2 X exec=ok fr=stop     out=40 rea=0
  B_enone_T0 n= 41 rep=1 O exec=ok fr=stop     out=42 rea=0
  B_enone_T0 n= 41 rep=2 X exec=ok fr=stop     out=211 rea=0
  B_enone_T0 n= 52 rep=1 O exec=ok fr=stop     out=48 rea=0
  B_enone_T0 n= 52 rep=2 O exec=ok fr=stop     out=48 rea=0
  B_enone_T0 n= 60 rep=1 O exec=ok fr=stop     out=57 rea=0
  B_enone_T0 n= 60 rep=2 O exec=ok fr=stop     out=57 rea=0
  B_enon

In [14]:
# 5B) 요약
L.summarize(resB, by_condition=True)

,condition,호출 수,호출 실패,finish=length,본문 비어있음,실행 성공률 %,서명 추출 %,정답률 %,값 일관성 %,의도 일관성 %,출력토큰 평균,사고토큰 평균,캐시hit 평균,fingerprint 수,비용 USD
0,B_enone_T0,20,0,0,0,100.0,100.0,70.0,87.5,100.0,78.6,0.0,9917.0,0,0.0226
1,B_enone_T0.2,20,0,0,0,100.0,100.0,60.0,75.0,85.0,83.2,0.0,19834.0,0,0.0048
2,B_enone_T1,20,0,0,0,100.0,100.0,60.0,75.0,95.0,66.5,0.0,19834.0,0,0.0046


## A vs B — T=1.0 접점 비교

In [ ]:
# 6) 같은 문항·같은 온도(1.0)에서 사고 유무만 다른 칸
import pandas as pd
a = resA[resA.condition == "A_emedium_T1"].copy()
b = resB[resB.condition == "B_enone_T1"].copy()
cmp = pd.concat([a.assign(갈래="A 사고중간"), b.assign(갈래="B 사고없음")])

piv = cmp.pivot_table(index=["n","dataset"], columns="갈래",
                      values=["correct","sig_key","answer_agent","completion_tokens"],
                      aggfunc="first")
piv

In [ ]:
# 7) 사고 토큰이 실제로 얼마나 쓰이는가
import pandas as pd
chk = pd.concat([
    resA[["condition","completion_tokens","reasoning_tokens","finish_reason","body_len"]],
    resB[["condition","completion_tokens","reasoning_tokens","finish_reason","body_len"]],
])
chk.groupby("condition").agg(
    출력=("completion_tokens","mean"), 사고=("reasoning_tokens","mean"),
    본문길이=("body_len","mean"), 잘림=("finish_reason", lambda s:(s=="length").sum()),
).round(1)

In [ ]:
# 8) 잘림 · 빈 본문 · 실행 실패 점검
import pandas as pd
for name, d in [("A", resA), ("B", resB)]:
    ok = d[d.call_error.fillna("")==""]
    print(f"[{name}] 호출 {len(d)} · 실패 {(d.call_error.fillna('')!='').sum()}"
          f" · length {(ok.finish_reason=='length').sum()}"
          f" · 빈본문 {(ok.body_len==0).sum()}"
          f" · 실행실패 {(~ok.exec_ok).sum()}")

# 사고가 예산을 다 먹은 사례
bad = pd.concat([resA, resB]).query("finish_reason=='length' or body_len==0")
if len(bad):
    i = bad.index[0]
    print("\n--- 사례 ---"); print(bad.loc[i,"raw"][:600])

## 판정 기준

| 항목 | A (사고 켬) | B (사고 끔) |
|---|---|---|
| `length` 비율 | >5% → 2000→4000 상향 | >5% → 300→512 상향 |
| 빈 본문 | >0 이면 즉시 원인 확인 | — |
| 실행 성공률 | <60% → 프롬프트 점검 | 동일 |
| 서명 추출 | <90% → 파서 보완 | 동일 |
| 온도 T=0 값 일관성 | — | 100%면 흔들림 없음 |